# Text conditioning by cross-attention on Tahoe cell states

Comparison method 3 of 3 for the cell-state modality. The condition enters the network as text,
through one cross-attention block, with no guidance at sampling time. The notebook loads (or
trains) the shared base diffusion model, freezes it, attaches one cross-attention block after the
MLP's input projection, trains only that block, and samples.

- Paper: Chen et al., *PixArt-α: Fast Training of Diffusion Transformer for Photorealistic
  Text-to-Image Synthesis*, ICLR 2024, [arXiv:2310.00426](https://arxiv.org/abs/2310.00426).
- Official code: [PixArt-alpha/PixArt-alpha](https://github.com/PixArt-alpha/PixArt-alpha),
  `MultiHeadCrossAttention` in `PixArt_blocks.py` and its zero-init in `PixArt.py`. Our version is
  `src/guidance/baselines/cross_attention.py`; its header lists what was kept and what changed,
  including the single-query-token form used here.
- The text is the only route to the condition: the backbone runs with the null label, and the
  block reads CLIP token embeddings of the pair's description (drug mechanism and targets, cell
  line name and organ). Same protocol as the TC-LoRA cell notebook, so the two arms compare
  directly.

How to run: top to bottom on a GPU. Run `cfg_cells.ipynb` first on the same machine; its
`checkpoints/diffusion_cfg/best.pt` is loaded here (or the path in `CELL_BASE_CKPT`). A `base.pt`
written by any cell notebook is also accepted. If neither exists, the base is trained here with
the same recipe.

## What the PixArt-α block does, and how it is used here

Notation: $x_k$ is the noisy cell state at step $k$ ($K = 1000$; the network receives $t = k/K$),
$\varepsilon_\theta$ the frozen noise-prediction MLP over 50 principal components, $c$ the
(cell line, drug) pair, $\varnothing$ the null label, $h(c) \in \mathbb R^{L \times 512}$ the frozen
CLIP token embeddings of the pair's description with padding mask $M$, and $\omega$ the
parameters trained here.

PixArt-α feeds text into a diffusion transformer through a cross-attention block in every
transformer block: the image tokens are the queries, the text tokens the keys and values, and
the result is added back as a plain residual. The output projection is zero-initialised, so a
freshly attached block is the identity and training starts from the unconditioned model:

$$Q = z W_q,\qquad [K\;\;V] = h(c)\, W_{kv},\qquad
\mathrm{Attn}(z, h(c)) = \operatorname{softmax}\!\Big(\frac{Q K^\top}{\sqrt d} + \log M\Big) V$$

$$z' = z + \mathrm{Attn}(z, h(c))\, W_o,\qquad W_o = 0 \text{ at initialisation}$$

Here one such block sits after the frozen MLP's input projection. The 256-wide hidden vector
$z \in \mathbb R^{256}$ is a single query token over the text tokens (deviation 7 in the header:
the softmax is still over the text, so the block still selects between words). Four heads. Only
$W_q, W_{kv}, W_o$ train (394k parameters), with the ordinary noise-prediction loss and the null
label, so the text is the model's only route to the condition:

$$\mathcal L(\omega) = \mathbb E_{k,\,x_0,\,\varepsilon}\Big[\big\|\varepsilon -
\varepsilon_{\theta,\omega}\big(x_k, k, \varnothing;\ h(c)\big)\big\|_2^2\Big]$$

Sampling is the plain ancestral sampler with the conditioned network, 250 evaluations, no
guidance. So this arm measures what the conditioning mechanism does on its own.

| PixArt-α (paper and code) | Here |
|---|---|
| `MultiHeadCrossAttention`: `q_linear` on image tokens, `kv_linear` on text tokens, one output `proj`, padding mask | `TextCrossAttention` in `cross_attention.py` |
| `proj` zero-initialised in `PixArt.initialize_weights` | `TextCrossAttention.proj`; the `allclose` check in *Attach cross-attention* |
| `x = x + cross_attn(x, y, mask)`: a plain residual, no norm, no gate | `_Conditioned`, wrapped around the stage named by `cfg.attach`, inside `TextConditioned` |
| T5-XXL caption tokens through a `y_embedder` MLP | CLIP token embeddings from `build_token_embeddings`, straight into `kv_linear` |
| cross-attention in every DiT block | one block after the input projection, one query token |
| not in PixArt-α | the null label, the wrong-text pass and matched validation (`COND`, `xa_val_wrong_text`, `validate`), the same protocol as the TC-LoRA notebook |

Differences from the paper. Their text encoder is T5-XXL (4096-dim); ours is CLIP, the encoder
every arm here uses, so the comparison isolates the mechanism. They put the block in every
transformer block and train a 600M-parameter model in stages on 25M images; we attach one block
to a frozen 0.7M MLP and train only that block on 791k cells. They sample with classifier-free
guidance at scale 4.5; we sample with none. Seven deviations are listed in the header of
`cross_attention.py`.

Why token embeddings and not a pooled vector: with a single key the softmax is identically 1 and
the block collapses to adding a constant. It would still condition, but it could not select
between words, which is the mechanism under test.

In [ ]:
# Colab bootstrap. Run once.
import importlib.util, subprocess, sys, os

for pkg in ("transformers", "datasets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

if importlib.util.find_spec("guidance") is None:
    # Inside a checkout (Jupyter on a server, cwd somewhere under the repo): walk up to its src/.
    from pathlib import Path as _P
    _src = next((p / "src" for p in [_P.cwd(), *_P.cwd().parents] if (p / "src" / "guidance").is_dir()), None)
    if _src is None:                                   # Colab: clone the repo next to the notebook
        if not os.path.exists("discrete-generative-models-proj-1"):
            subprocess.run(["git", "clone", "-q",
                            "https://github.com/imperorrp/discrete-generative-models-proj-1.git"],
                           check=True)
        _src = "discrete-generative-models-proj-1/src"
    sys.path.insert(0, str(_src))
    print("guidance package from:", _src)

import torch
from guidance.baselines.cross_attention import TextConditioned

def _git_sha():
    for d in ("discrete-generative-models-proj-1", "."):
        try:
            return subprocess.check_output(["git", "-C", d, "rev-parse", "--short", "HEAD"],
                                           text=True, stderr=subprocess.DEVNULL).strip()
        except Exception:
            pass
    return "unknown"

GIT_SHA = _git_sha()
print("code version:", GIT_SHA)
print("device:", "cuda" if torch.cuda.is_available()
      else "CPU  <-- Runtime > Change runtime type > GPU")

## Config

All settings in one place. Any field can be set from the environment as `CFG_<NAME>`, for
example `CFG_LIMIT=5000 CFG_BASE_EPOCHS=1` for a smoke run, so a headless run never needs a cell
edited.

In [ ]:
import math, time, json
from dataclasses import dataclass, asdict, fields
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

@dataclass
class Cfg:
    seed: int = 42                 # training randomness, as cfg_cells
    split_seed: int = 42           # shuffling only; the Tahoe split is fixed by the data files
    batch_size: int = 256
    base_epochs: int = 50
    xattn_epochs: int = 10
    lr_base: float = 2e-4
    lr_schedule: str = "constant"     # cosine anneals to zero; "constant" is the cell base's recipe
    amp: bool = True               # fp16 autocast on CUDA; the cell base's recipe
    lr_xattn: float = 1e-4
    K: int = 1000                  # diffusion training steps
    sample_steps: int = 250        # sampling steps, as the cell base's own notebook
    eta: float = 1.0               # ancestral DDPM, as the cell base's own notebook
    clip_x0: float = -1.0          # clamp of the predicted clean state: -1 = per-PC training range (cells) or [-1, 1] (images); 0 = off; c > 0 = [-c, c]
    heads: int = 4
    attach: str = "input_proj"     # MLP stage whose output the text conditions (the input projection)
    label_path: bool = False       # False: backbone sees the null label; text is the only condition
    reuse_base: bool = True        # load base.pt if present instead of retraining
    limit: int = 0                 # >0 truncates the training set, for smoke tests
    samples_per_condition: int = 400
    n_eval_conditions: int = 6     # held-out pairs evaluated: the largest pair of each of this many drugs
    condition: str = 'pair'        # pair (drug + cell line) | drug; see guidance.data.cells
    dose: float = -1.0             # -1 keeps every dose; 0.05, 0.5 or 5 keeps that dose (uM) plus controls

cfg = Cfg()

# Any field can be overridden from the environment, e.g. CFG_BASE_EPOCHS=1 CFG_LIMIT=5000, so
# headless runs (nbconvert, a server, a different GPU) never need a cell edited.
for _f in fields(Cfg):
    _v = os.environ.get(f"CFG_{_f.name.upper()}")
    if _v is not None:
        _cur = getattr(cfg, _f.name)
        setattr(cfg, _f.name, tuple(float(t) for t in _v.split(",")) if isinstance(_cur, tuple)
                else type(_cur)(_v) if not isinstance(_cur, bool) else _v.lower() in ("1", "true", "yes"))
        print(f"cfg.{_f.name} <- {getattr(cfg, _f.name)!r} (from CFG_{_f.name.upper()})")
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
print(DEV, asdict(cfg))

## Data

Tahoe-100M subset: 25 cell lines x 100 drugs at three doses, plus DMSO controls; 991,899 cells.
Each cell is 2000 highly variable genes reduced to 50 principal components. Normalisation, gene
selection, scaling and PCA were fitted on the training split only
(`notebooks/tahoe/tahoe_colab_downloader_with_pca.ipynb`). About 20% of (cell line, drug) pairs
are held out: every validation cell comes from a pair never seen in training, while its line and
its drug each appear in other training pairs, so validation loss already measures generalisation
to unseen combinations. `x` is the 50-vector, scaled so its overall variance is 1.

The condition id is the (cell line, drug) pair, 2525 ids. The backbone turns it into a drug
embedding plus a cell-line embedding, so an unseen pair is a seen drug plus a seen line. Each
pair's text is the drug's mechanism of action and targets plus the cell line's name and organ,
for example "Binimetinib, mek inhibitor targeting MAP2K1, MAP2K2, in HepG2/C3A liver cells",
written from the public metadata before any result was seen.

The loader looks for the unpacked `25lines_100drugs_pca_data_v2.zip` next to the notebook, under
`data/` in the repo, or on Colab and Drive; set `TAHOE_PCA_DIR` if it is somewhere else.

In [ ]:
from guidance.data.cells import make_loaders

train_loader, val_loader, meta = make_loaders(
    batch_size=cfg.batch_size, seed=cfg.split_seed, condition=cfg.condition,
    dose=None if cfg.dose < 0 else cfg.dose, limit=cfg.limit)

xb, cb = next(iter(train_loader))
print("x", tuple(xb.shape), f"rms {float(xb.pow(2).mean().sqrt()):.2f}", "| cond", tuple(cb.shape))
print(f"train cells {meta['n_train']:,} | val cells {meta['n_val']:,} | "
      f"conditions {meta['n_conditions']} ({meta['condition']}) | split: {meta['split']}")
print("example text:", meta["cond_text"][1])

## Frozen text encoder: token embeddings

The TC-LoRA arm uses CLIP's pooled 512-vector. Cross-attention needs the per-token embeddings and
the padding mask (see above). If CLIP cannot be loaded the cell falls back to fixed random tokens
so the notebook still runs; the manifest records `clip_fallback`, and a fallback run is not a
result.

In [ ]:
def build_token_embeddings(texts, device):
    '''Returns (tokens (N, L, D), mask (N, L) bool, is_fallback).'''
    try:
        from transformers import CLIPTokenizer, CLIPTextModel, logging as hf_logging
        hf_logging.set_verbosity_error()          # silence the vision_model.* UNEXPECTED report
        name = "openai/clip-vit-base-patch32"
        tok = CLIPTokenizer.from_pretrained(name)
        enc = CLIPTextModel.from_pretrained(name).to(device).eval()
        for p in enc.parameters():
            p.requires_grad_(False)
        b = tok(texts, padding=True, truncation=True, return_tensors="pt")
        with torch.no_grad():
            hidden = enc(**{k: v.to(device) for k, v in b.items()}).last_hidden_state
        return hidden, b["attention_mask"].to(device).bool(), False
    except Exception as e:
        print(f"CLIP unavailable ({type(e).__name__}); using random fallback tokens.")
        g = torch.Generator().manual_seed(0)
        toks = torch.randn(len(texts), 8, 512, generator=g).to(device)
        return toks, torch.ones(len(texts), 8, dtype=torch.bool, device=device), True

TEXT_TOK, TEXT_MASK, IS_FALLBACK = build_token_embeddings(meta["cond_text"], DEV)
TEXT_DIM = TEXT_TOK.shape[-1]
print("token embeddings:", tuple(TEXT_TOK.shape), "| real tokens per prompt:",
      TEXT_MASK.sum(1).tolist()[:5], "... | fallback:", IS_FALLBACK)

## Backbone

The MLP from the teammate's `Tahoe_PCA_Diffusion_CFG.ipynb`, kept as written: the drug and the
cell line are embedded separately, summed with the time embedding, and that vector enters every
residual block. `CellDiffusionMLP` wraps it so the notebooks can pass one pair id and `t` in
[0, 1]: `COND_TABLE` maps the pair id to (drug id, line id), with an extra last row for the null
condition (both ids null), and `t` is scaled back to the integer step the time embedding was
written for.

`MID_CHANNELS` is the width of the stage the text block attaches to, its one query token.

In [ ]:
def timestep_embedding(t, dim):
    half = dim // 2
    freqs = torch.exp(-math.log(10_000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([args.cos(), args.sin()], dim=-1)

# ---- The model of notebooks/tahoe/Tahoe_PCA_Diffusion_CFG.ipynb, a teammate's classifier-free-
# guidance base for these cells, kept verbatim: drug and cell line are embedded separately and
# summed with the time embedding into one vector that enters every residual block.
class SinusoidalTimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        half = dim // 2
        freqs = torch.exp(-math.log(10000.0) * torch.arange(half) / max(half - 1, 1))
        self.register_buffer("freqs", freqs, persistent=False)

    def forward(self, t):
        args = t.float()[:, None] * self.freqs[None, :].to(t.device)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class ResidualBlock(nn.Module):
    def __init__(self, dim, cond_dim):
        super().__init__()
        self.norm = nn.LayerNorm(dim)
        self.fc1 = nn.Linear(dim + cond_dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.act = nn.SiLU()

    def forward(self, x, cond):
        h = torch.cat([self.norm(x), cond], dim=-1)
        h = self.fc2(self.act(self.fc1(h)))
        return x + h


class DiffusionMLP(nn.Module):
    def __init__(self, data_dim, n_drugs, n_lines, time_dim=128, cond_dim=128, hidden_dim=256, n_blocks=4):
        super().__init__()
        self.data_dim = data_dim
        self.time_embed = SinusoidalTimeEmbedding(time_dim)
        self.time_mlp = nn.Sequential(nn.Linear(time_dim, cond_dim), nn.SiLU(), nn.Linear(cond_dim, cond_dim))
        self.drug_embed = nn.Embedding(n_drugs + 1, cond_dim)   # last row is the null/unconditional id
        self.line_embed = nn.Embedding(n_lines + 1, cond_dim)   # last row is the null/unconditional id
        self.input_proj = nn.Linear(data_dim, hidden_dim)
        self.blocks = nn.ModuleList([ResidualBlock(hidden_dim, cond_dim) for _ in range(n_blocks)])
        self.output_proj = nn.Linear(hidden_dim, data_dim)

    def forward(self, x, t, drug_id, line_id):
        cond = self.time_mlp(self.time_embed(t)) + self.drug_embed(drug_id) + self.line_embed(line_id)
        h = self.input_proj(x)
        for block in self.blocks:
            h = block(h, cond)
        return self.output_proj(h)


# ---- Our wrapper. The notebooks pass one cond_id and t in [0, 1]. The loader's table turns the
# id into (drug_id, line_id); the extra last row is the null condition, both ids null, which is
# their joint dropout. t is scaled back to the integer step their time embedding was written for.
COND_TABLE = torch.cat([meta["cond_ids_table"],
                        torch.tensor([[meta["n_drugs"], meta["n_lines"]]])]).to(DEV)
assert int(COND_TABLE.min()) >= 0, "this backbone needs condition = 'pair' or 'drug'"


class CellDiffusionMLP(DiffusionMLP):
    def __init__(self):
        super().__init__(meta["shape"][0], n_drugs=meta["n_drugs"], n_lines=meta["n_lines"])
        self.null = meta["n_conditions"]

    def forward(self, x, t, cond_ids=None):
        if cond_ids is None:
            cond_ids = torch.full((x.shape[0],), self.null, device=x.device, dtype=torch.long)
        ids = COND_TABLE[cond_ids]
        return super().forward(x, t * cfg.K, ids[:, 0], ids[:, 1])

    def forward_ids(self, x, t, drug_id, line_id):
        '''Direct (drug, line) ids, for partially conditioned branches such as line known, drug null.'''
        return super().forward(x, t * cfg.K, drug_id, line_id)


def make_backbone():
    return CellDiffusionMLP()

# Only the trunk is adapted by TC-LoRA: the input projection and the residual blocks' two
# linear layers each. Time MLP, the two embeddings and the output projection stay untouched.
SKIP_NAMES = ("output_proj", "time_mlp", "drug_embed", "line_embed")

torch.manual_seed(cfg.seed)      # model init and training randomness follow `seed`
net = make_backbone().to(DEV)
print(f"backbone params: {sum(p.numel() for p in net.parameters()):,}")

MID_CHANNELS = 256                   # width of the stage named by cfg.attach; the block's one query token

## Diffusion: schedule, noising, sampler

The cosine schedule as diffusers builds it (`squaredcos_cap_v2`), which is the schedule the cell
base was trained with. `add_noise` draws a random step and a noise vector for training.
`ddim_sample` is the ancestral sampler (`eta = 1`, 250 steps) written out in plain PyTorch, the
same update as the diffusers `DDPMScheduler` step used in `cfg_cells`, plus a clamp on the
predicted clean state; the code comment explains why the clamp is there and how to turn it off.

In [ ]:
def cosine_alpha_bar(K, s=0.008, device="cpu"):
    steps = torch.arange(K + 1, dtype=torch.float32, device=device) / K
    f = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
    return (f / f[0])[1:].clamp(1e-5, 0.9999)

def cosine_alpha_bar_diffusers(K, max_beta=0.999, device="cpu"):
    '''diffusers "squaredcos_cap_v2": the same cosine curve turned into betas clipped at
    max_beta, then a cumulative product. The schedule the cell base (cfg_cells) was trained with.'''
    steps = torch.arange(K + 1, dtype=torch.float64, device=device) / K
    f = torch.cos((steps + 0.008) / 1.008 * math.pi / 2) ** 2
    betas = (1 - f[1:] / f[:-1]).clamp(max=max_beta)
    return torch.cumprod(1 - betas, 0).float()

ABAR = cosine_alpha_bar(cfg.K, device=DEV) if len(meta["shape"]) == 3 else cosine_alpha_bar_diffusers(cfg.K, device=DEV)
NULL_ID = meta["n_conditions"]

# ---- Clamp on the predicted clean state x0_hat: [-1, 1] for images; for cells, each principal
# component's training range, which an in-range estimate never touches. Why: at the first sampling
# step abar is 2e-9, so x0_hat = (x - sqrt(1 - abar) eps) / sqrt(abar) multiplies any error in eps
# by about 20,000, and a base that is slightly off there sends the trajectory out of range for
# good. cfg_cells samples without a clamp and its base is precise enough; here the clamp is a
# safety net applied to every row alike, and the on-clamp fraction is reported. CFG_CLIP_X0=0
# turns it off.
def x0_bounds(c):
    '''(lo, hi) tensors that broadcast over a batch, or None for no clamp. See cfg.clip_x0.'''
    if c == 0:
        return None
    if c > 0:
        return torch.tensor(-float(c), device=DEV), torch.tensor(float(c), device=DEV)
    if len(meta["shape"]) == 3:
        return torch.tensor(-1.0, device=DEV), torch.tensor(1.0, device=DEV)
    _xtr = train_loader.dataset.x
    return _xtr.min(0).values.to(DEV), _xtr.max(0).values.to(DEV)

def clamp_x0(x0, bounds):
    return x0 if bounds is None else torch.maximum(torch.minimum(x0, bounds[1]), bounds[0])

def saturation(v, bounds=None):
    '''Fraction of coordinates sitting on the clamp.'''
    bounds = X0_BOUNDS if bounds is None else bounds
    if bounds is None:
        return float("nan")
    return ((v <= bounds[0] + 1e-6) | (v >= bounds[1] - 1e-6)).float().mean().item()

X0_BOUNDS = x0_bounds(cfg.clip_x0)
X0_CLAMP_DESC = ("off" if X0_BOUNDS is None else
                 "per-PC training range" if cfg.clip_x0 < 0 and len(meta["shape"]) == 1 else
                 f"[-{X0_BOUNDS[1].max().item():g}, {X0_BOUNDS[1].max().item():g}]")
print("x0 clamp:", X0_CLAMP_DESC)
print(f"abar at the first sampling step {ABAR[-1].item():.1e}: x0_hat amplifies the denoiser's residual by {1 / ABAR[-1].sqrt().item():,.0f}")


def add_noise(x0, gen=None):
    '''gen: pass a seeded torch.Generator to get the same (k, eps) every call.'''
    n = x0.shape[0]
    k = torch.randint(0, cfg.K, (n,), device=x0.device, generator=gen)
    eps = torch.randn(x0.shape, device=x0.device, generator=gen)
    ab = ABAR[k].view(n, *([1] * (x0.dim() - 1)))
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps, k, eps


def drop_labels(y, p_drop=0.1):
    '''Condition dropout creates the null branch that guidance needs later.'''
    keep = torch.rand(y.shape[0], device=y.device) >= p_drop
    return torch.where(keep, y, torch.full_like(y, NULL_ID))


@torch.no_grad()
def ddim_sample(eps_fn, n, shape, steps, clip_x0=None, n_snapshots=0, x_init=None, eta=None, generator=None):
    '''eps_fn(x, t) -> predicted noise.

    Convention, shared by every arm in this project: clip the predicted x0, then recompute
    eps from the clipped x0 before stepping (as TFG's official sampler does). Keeping the
    original eps is a different update whenever the clamp engages.

    clip_x0: None uses the notebook's X0_BOUNDS; 0 disables; c > 0 clamps to [-c, c].
    x_init: pass the same starting noise to two calls for a paired comparison; with eta > 0
    also pass identically seeded generators, or the per-step noise differs.
    eta: 0 deterministic DDIM (images), 1 ancestral DDPM (cells, as the base's own notebook).
    n_snapshots > 0 also returns a list of (timestep, state) pairs for a trajectory figure:
    the starting noise, then the model's clean estimate x0_hat at that many evenly spaced
    steps. The last pair is the finished sample.
    '''
    bounds = X0_BOUNDS if clip_x0 is None else x0_bounds(clip_x0)
    idx = torch.linspace(cfg.K - 1, 0, steps).long().to(DEV)
    keep = set(torch.linspace(1, steps - 1, max(n_snapshots, 1)).long().tolist())
    eta = cfg.eta if eta is None else eta
    x = torch.randn(n, *shape, device=DEV, generator=generator) if x_init is None else x_init.clone()
    snaps = [(cfg.K, x.cpu().clone())] if n_snapshots else []
    for i, k in enumerate(idx):
        t = (k.float() / cfg.K).repeat(n)
        eps = eps_fn(x, t)
        ab = ABAR[k]
        ab_prev = ABAR[idx[i + 1]] if i + 1 < len(idx) else torch.ones_like(ab)
        x0 = clamp_x0((x - (1 - ab).sqrt() * eps) / ab.sqrt(), bounds)
        eps = (x - ab.sqrt() * x0) / (1 - ab).sqrt()          # recompute from (the clipped) x0, as TFG does
        # eta = 0 is deterministic DDIM; eta = 1 is ancestral DDPM (DDIM paper eq. 16), with fresh
        # noise at every step drawn from `generator` so paired rows stay paired.
        sigma = eta * ((1 - ab_prev) / (1 - ab) * (1 - ab / ab_prev)).sqrt()
        x = ab_prev.sqrt() * x0 + (1 - ab_prev - sigma ** 2).clamp(min=0).sqrt() * eps
        if eta > 0:
            x = x + sigma * torch.randn(x.shape, device=DEV, generator=generator)
        if n_snapshots and i in keep:
            # x0_hat, not x_t: the noisy state looks like noise until the last few steps,
            # while the clean estimate shows what the model believes at every step.
            snaps.append((int(k), x0.cpu().clone()))
    return (x, snaps) if n_snapshots else x

## Base model: load the `cfg_cells` base, or `base.pt`, or train

In order of preference: a `base.pt` written by any cell notebook in this folder; the `cfg_cells`
checkpoint `checkpoints/diffusion_cfg/best.pt` (or the path in `CELL_BASE_CKPT`), which is the
base every cell notebook should share; otherwise the base is trained here with the same recipe.
A reused checkpoint is checked against this run's split, schedule and backbone first. The base's
per-epoch history travels with the checkpoint, so the training-curve figure shows the curve of
whichever run trained it, and the manifest records where the base came from.

Training and validation use different losses on purpose. Training drops the label 10% of the
time so the null branch exists (and so the null-label setting is in-distribution for the frozen
model). Validation does not: it scores the model on the same fixed noise and timesteps every
epoch, and the adapter is validated the same way, so the two numbers are comparable.

In [ ]:
VAL_SEED = cfg.seed + 999


def run_epochs(model, params, n_epochs, lr, tag, train_fn, val_fn, ckpt):
    '''Trains with train_fn, validates with val_fn on fixed noise, keeps the best checkpoint.'''
    # cfg.lr_schedule: "cosine" anneals to zero over the run (images); "constant" holds lr, the
    # cell base's recipe. cfg.amp: fp16 autocast with a grad scaler on CUDA, also the cell base's
    # recipe; the loss itself is computed in float32 either way.
    opt = torch.optim.AdamW(params, lr=lr)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(n_epochs, 1)) if cfg.lr_schedule == "cosine"
             else torch.optim.lr_scheduler.LambdaLR(opt, lambda _: 1.0))
    use_amp = bool(cfg.amp) and DEV.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
    hist, best, best_ep = [], float("inf"), 0
    for ep in range(1, n_epochs + 1):
        model.train(); tot = num = 0; t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(DEV), y.to(DEV)
            with torch.autocast(device_type="cuda", enabled=use_amp):
                loss = train_fn(x, y)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt); scaler.update()
            tot += loss.item() * x.shape[0]; num += x.shape[0]
        sched.step()

        val = validate(model, val_fn)
        is_best = val < best
        if is_best:
            best, best_ep = val, ep
            torch.save({"state": model.state_dict(), "epoch": ep, "val": val}, ckpt)
        hist.append({"epoch": ep, "train": tot / num, "val": val, "best": is_best,
                     "seconds": time.time() - t0})
        print(f"{tag} epoch {ep:3d}  train {tot/num:.4f}  val {val:.4f}"
              f"{'  *best' if is_best else ''}  ({time.time()-t0:.0f}s)")
    print(f"{tag}: best val {best:.4f} at epoch {best_ep}, saved to {ckpt}")
    return hist


@torch.no_grad()
def validate(model, val_fn):
    '''Same examples, same timesteps, same noise on every call -- for any model.'''
    model.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    tot = num = 0
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        tot += val_fn(x, y, gen).item() * x.shape[0]; num += x.shape[0]
    return tot / num


def base_train(x, y):
    xk, k, eps = add_noise(x)
    return F.mse_loss(net(xk, k.float() / cfg.K, drop_labels(y)), eps)


def base_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(net(xk, k.float() / cfg.K, y), eps)     # no dropout


def hist_from_cfg_ckpt(ck):
    '''cfg_cells' checkpoint carries its per-epoch history. Recorded as it was (its own noise
    draws), without seconds, which are not this run's time.'''
    h = ck.get("history") or []
    return ([{"epoch": r["epoch"], "train": r["train_loss"], "val": r["val_loss"], "best": r["epoch"] == ck["epoch"]}
             for r in h]
            or [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val_loss"], "best": True}])


# What a base.pt must agree on before it can be reused. A checkpoint trained under a
# different split would put its own training cells into this run's validation set.
CKPT_META = {"split_seed": cfg.split_seed, "n_conditions": meta["n_conditions"],
             "shape": list(meta["shape"]), "K": cfg.K, "schedule": "cosine",
             "backbone_params": sum(p.numel() for p in net.parameters())}

if cfg.reuse_base and os.path.exists("base.pt"):
    ck = torch.load("base.pt", map_location=DEV, weights_only=False)
    saved = ck.get("meta")
    if saved is None:
        print("WARNING: base.pt carries no setup metadata (older run); split and schedule "
              "cannot be verified. Treat any comparison against it with care.")
    else:
        mismatch = {k: (saved.get(k), v) for k, v in CKPT_META.items() if saved.get(k) != v}
        assert not mismatch, f"base.pt was trained under a different setup: {mismatch}"
    net.load_state_dict(ck["state"])
    base_hist = ck.get("history") or [{"epoch": ck["epoch"], "train": float("nan"), "val": ck["val"], "best": True}]
    BASE_SOURCE = f"base.pt (from {ck.get('source', 'an earlier cell notebook run')})"
    print(f"loaded base.pt (epoch {ck['epoch']}, val {ck['val']:.4f}); skipping base training")
elif cfg.reuse_base and len(meta["shape"]) == 1 and (_cfg_ckpt := next(
        (p for p in [os.environ.get("CELL_BASE_CKPT"), "checkpoints/diffusion_cfg/best.pt",
                     "../checkpoints/diffusion_cfg/best.pt", "../../checkpoints/diffusion_cfg/best.pt"]
         if p and os.path.exists(p)), None)):
    # The base trained by cfg_cells.ipynb: same model class, so the weights load directly, and
    # every cell notebook sits on the same weights. Its validation loss was computed with that
    # notebook's noise draws, so it is reported as recorded, not recomputed.
    ck = torch.load(_cfg_ckpt, map_location=DEV, weights_only=False)
    assert ck["meta"]["n_params"] == sum(p.numel() for p in net.parameters()) and \
        ck["meta"]["num_train_timesteps"] == cfg.K, f"{_cfg_ckpt} does not match this backbone"
    net.load_state_dict(ck["model"])
    base_hist = hist_from_cfg_ckpt(ck)
    BASE_SOURCE = _cfg_ckpt
    torch.save({"state": net.state_dict(), "epoch": ck["epoch"], "val": ck["val_loss"], "meta": CKPT_META,
                "history": base_hist, "source": _cfg_ckpt}, "base.pt")
    print(f"loaded the cfg_cells base from {_cfg_ckpt} (epoch {ck['epoch']}, val {ck['val_loss']:.4f}); "
          f"saved as base.pt; skipping base training")
else:
    base_hist = run_epochs(net, list(net.parameters()), cfg.base_epochs, cfg.lr_base,
                           "base", base_train, base_val, "base.pt")
    BASE_SOURCE = "trained in this notebook"
    ck = torch.load("base.pt", map_location=DEV, weights_only=False)
    ck["meta"] = CKPT_META; ck["source"] = "text_xattn_cells"
    ck["history"] = [{k: v for k, v in h.items() if k != "seconds"} for h in base_hist]
    torch.save(ck, "base.pt")
print("base source:", BASE_SOURCE)

## Attach cross-attention and train only that block

The backbone is frozen. One `TextCrossAttention` block is inserted after the stage named by
`cfg.attach`; it attends from the hidden vector to the text tokens and adds the result back. Its
output projection starts at zero (PixArt-α's initialisation), so the first assertion, adapted
model equals base at step zero, is what makes this a controlled comparison.

`COND(y)` is the label the backbone sees: the true label if `cfg.label_path`, otherwise the null
token. With the null token, the text is the model's only knowledge of the condition.

The next cell trains the block with the ordinary noise-prediction loss and reloads the best
checkpoint.

In [ ]:
ckpt = torch.load("base.pt", map_location=DEV)
base_frozen = make_backbone().to(DEV)
base_frozen.load_state_dict(ckpt["state"])
base_frozen.eval()

def COND(y):
    '''The label the frozen backbone sees. Null by default, so the class can only reach the
    model through the text. cfg.label_path = True gives the backbone the true label.'''
    return y if cfg.label_path else torch.full_like(y, NULL_ID)

probe_x, probe_y = xb[:4].to(DEV), cb[:4].to(DEV)
probe_t = torch.rand(4, device=DEV)
with torch.no_grad():
    ref = base_frozen(probe_x, probe_t, COND(probe_y)).clone()

# PixArtBlock.forward: x = x + cross_attn(x, text), here after the U-Net stage named by
# cfg.attach. Queries come from the feature map, keys and values from the text tokens.
xa = TextConditioned(base_frozen, attach=cfg.attach, channels=MID_CHANNELS,
                     text_dim=TEXT_DIM, heads=cfg.heads).to(DEV)

# PixArt.initialize_weights: the output projection starts at zero, so the block is the
# identity at this point and everything it later does is measured from the base model.
with torch.no_grad():
    got = xa(probe_x, probe_t, TEXT_TOK[probe_y], TEXT_MASK[probe_y], cond_ids=COND(probe_y))
assert torch.allclose(got, ref, atol=1e-5), (got - ref).abs().max().item()
print("no-op at init confirmed")

assert all(".attn." in n for n, p in xa.named_parameters() if p.requires_grad)
# The adapter now lives INSIDE base_frozen, so count frozen params by requires_grad, not
# by module membership -- otherwise the adapter is counted on both sides.
N_FROZEN = sum(p.numel() for p in base_frozen.parameters() if not p.requires_grad)
print(f"trainable: {xa.n_trainable():,} params, all in the cross-attention block")
print(f"frozen:    {N_FROZEN:,} params")
print(f"label path: {'ON (true label)' if cfg.label_path else 'OFF (null label; text only)'}")

In [ ]:
def xa_train(x, y):
    # The ordinary noise-prediction loss. Only the attention block has requires_grad; the
    # U-Net and CLIP stay fixed. No guidance anywhere: the text acts only through attention.
    xk, k, eps = add_noise(x)
    return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_TOK[y], TEXT_MASK[y], cond_ids=COND(y)), eps)


def xa_val(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_TOK[y], TEXT_MASK[y], cond_ids=COND(y)), eps)


xa_hist = run_epochs(xa, [p for p in xa.parameters() if p.requires_grad],
                     cfg.xattn_epochs, cfg.lr_xattn, "xattn", xa_train, xa_val, "xattn.pt")

best = torch.load("xattn.pt", map_location=DEV)["state"]
xa.load_state_dict(best)
xa.eval()
print("restored best cross-attention block")

## Validation losses on identical noise, and the wrong-text check

Three validation MSEs on the same examples, timesteps and noise:

- base, same label path: what the adapted model starts from (with the label path off, this is
  the unconditional model);
- base, true label: what knowing the pair is worth when it comes in through the label embedding;
- cross-attention: the arm under test.

Then the diagnostic: the same validation pass with the wrong text. Labels are shifted by a fixed
offset, so every example gets a different pair, and the shift uses no random state, so the
correct and wrong passes see identical noise. Averaged over three offsets. (A within-batch
permutation would not do: the validation set is ordered by pair, so most permuted labels would
land on the same pair.)

Read the result as the average loss sensitivity to the text. A small difference is a finding,
not proof that the text is ignored: it could matter only at some noise levels, or only in the
samples, which the next cells check. Base-vs-adapter differences include the adaptation itself;
correct-vs-wrong text within the adapted model is what isolates the text.

In [ ]:
def frozen_val_same_path(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, COND(y)), eps)


def frozen_val_label(x, y, gen):
    xk, k, eps = add_noise(x, gen)
    return F.mse_loss(base_frozen(xk, k.float() / cfg.K, y), eps)


def xa_val_wrong_text(offset):
    '''Wrong text via a fixed label shift: always a different class, never touches gen.
    Only the text changes. Noise, timesteps and the backbone's label input are identical to
    the correct-text pass, so any loss difference is the text.'''
    def fn(x, y, gen):
        xk, k, eps = add_noise(x, gen)
        y_wrong = (y + offset) % meta["n_conditions"]
        return F.mse_loss(xa(xk, k.float() / cfg.K, TEXT_TOK[y_wrong], TEXT_MASK[y_wrong],
                             cond_ids=COND(y)), eps)
    return fn


WRONG_OFFSETS = (1, 37, 73)

VAL_BASE_SAME = validate(base_frozen, frozen_val_same_path)
VAL_BASE_LABEL = validate(base_frozen, frozen_val_label)
VAL_XA = validate(xa, xa_val)
VAL_XA_WRONG_EACH = [validate(xa, xa_val_wrong_text(o)) for o in WRONG_OFFSETS]
VAL_XA_WRONG = sum(VAL_XA_WRONG_EACH) / len(VAL_XA_WRONG_EACH)

print(f"base, same label path as xattn : {VAL_BASE_SAME:.4f}")
print(f"base, true label (reference)   : {VAL_BASE_LABEL:.4f}")
print(f"xattn, correct text            : {VAL_XA:.4f}   ({(VAL_BASE_SAME - VAL_XA) / VAL_BASE_SAME:+.1%} vs base)")
print(f"xattn, wrong text (mean of {len(WRONG_OFFSETS)}) : {VAL_XA_WRONG:.4f}   "
      f"per offset {[round(v, 4) for v in VAL_XA_WRONG_EACH]}")
TEXT_EFFECT = (VAL_XA_WRONG - VAL_XA) / VAL_XA
print()
print(f"text effect (wrong - correct) / correct = {TEXT_EFFECT:+.2%}")
if abs(TEXT_EFFECT) < 0.005:
    print("little average loss sensitivity to the text at this training budget. "
          "Check the per-noise-level breakdown and the sample comparison below before concluding.")

## Text sensitivity by noise level

The number above averages over the whole trajectory. This bins the correct-vs-wrong loss gap by
noise level, on the same fixed noise.

In [ ]:
@torch.no_grad()
def loss_by_noise_level(offset, n_bins=5):
    xa.eval()
    gen = torch.Generator(device=DEV).manual_seed(VAL_SEED)
    edges = torch.linspace(0, cfg.K, n_bins + 1, device=DEV)
    sums = torch.zeros(n_bins, 2, device=DEV); counts = torch.zeros(n_bins, device=DEV)
    for x, y in val_loader:
        x, y = x.to(DEV), y.to(DEV)
        xk, k, eps = add_noise(x, gen)
        t = k.float() / cfg.K
        yw = (y + offset) % meta["n_conditions"]
        e_ok = ((xa(xk, t, TEXT_TOK[y], TEXT_MASK[y], cond_ids=COND(y)) - eps) ** 2).flatten(1).mean(1)
        e_wr = ((xa(xk, t, TEXT_TOK[yw], TEXT_MASK[yw], cond_ids=COND(y)) - eps) ** 2).flatten(1).mean(1)
        b = torch.bucketize(k.float(), edges[1:-1])
        sums.index_add_(0, b, torch.stack([e_ok, e_wr], 1)); counts.index_add_(0, b, torch.ones_like(b, dtype=torch.float))
    return edges.cpu(), (sums / counts[:, None]).cpu()

edges, per_bin = loss_by_noise_level(WRONG_OFFSETS[0])
print(f"{'noise level (k)':>18} {'correct':>9} {'wrong':>9} {'gap':>8}")
TEXT_EFFECT_BY_BIN = []
for i in range(len(per_bin)):
    ok, wr = per_bin[i].tolist()
    TEXT_EFFECT_BY_BIN.append((wr - ok) / ok)
    print(f"{int(edges[i]):>7}-{int(edges[i+1]):<9} {ok:>9.4f} {wr:>9.4f} {(wr-ok)/ok:>+8.2%}")

## Training curves

Train and validation loss per epoch, with the selected checkpoint marked (lowest validation
loss). The base panel shows the history that came with the loaded checkpoint; its source is in
the title. A checkpoint that carried no history shows one point.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, (name, hist) in zip(ax, [("base", base_hist), ("cross-attention", xa_hist)]):
    ep = [h["epoch"] for h in hist]
    if len(hist) > 1:
        if all(math.isfinite(h["train"]) for h in hist):
            a.plot(ep, [h["train"] for h in hist], label="train")
        a.plot(ep, [h["val"] for h in hist], label="val")
    else:                                   # one epoch, or a checkpoint that carried no history
        a.plot(ep, [h["val"] for h in hist], "o", label="val")
    bst = min(hist, key=lambda h: h["val"])
    a.axvline(bst["epoch"], ls="--", c="grey", lw=1)
    a.scatter([bst["epoch"]], [bst["val"]], c="red", zorder=5,
              label=f"selected (epoch {bst['epoch']})")
    a.set_title(f"{name}: best val {bst['val']:.4f}" + (f"\n{BASE_SOURCE}" if name == "base" else ""), fontsize=9)
    a.set_xlabel("epoch"); a.set_ylabel("noise-prediction MSE"); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Samples: same starting noise, three rows, six held-out pairs

The evaluation pairs are the largest held-out (cell line, drug) pair of each of the six most
populous drugs, one pair per drug, so the wrong text below is always a different drug (the six
largest pairs outright are five cell lines of one drug). `samples_per_condition` samples per
pair, all from the same starting noise and identically seeded generators: the base with the null
label, cross-attention with the correct description, and cross-attention with the next pair's description. Any
difference between the last two is the text.

The cell prints a scale check first: the total variance of each row over that of the real
validation cells. A row far above 1 has left the data range and nothing computed on it later
means anything. Then the first two principal components are plotted. The axes are
fixed by the real cells and shared by all panels; a row with points outside them gets that share
written in its title.

In [ ]:
# Evaluation pairs: the largest held-out pair of each of the n_eval_conditions most populous
# drugs, PER samples each; every metric is per pair. One pair per drug so the wrong target is a
# different drug (the six largest pairs outright are five lines of one drug). Ties in count are
# broken by (line, drug) id so every cell notebook picks the same pairs.
counts = torch.bincount(val_loader.dataset.cond, minlength=meta["n_conditions"])
counts[meta["cond_ids_table"][:, 0] == 0] = 0                 # drop DMSO conditions
EVAL_CONDS, _seen = [], set()
for c in sorted(range(meta["n_conditions"]), key=lambda c: (-int(counts[c]), int(meta["cond_ids_table"][c, 1]), int(meta["cond_ids_table"][c, 0]))):
    d = int(meta["cond_ids_table"][c, 0])
    if counts[c] > 0 and d not in _seen:
        EVAL_CONDS.append(c); _seen.add(d)
    if len(EVAL_CONDS) == cfg.n_eval_conditions:
        break
assert len(EVAL_CONDS) >= 2, "need at least two drugs for a wrong-text row"
PER = cfg.samples_per_condition
y_eval = torch.tensor(EVAL_CONDS, device=DEV).repeat_interleave(PER)
y_wrong = y_eval.roll(PER)                        # each block gets the next pair's text: a different drug by construction
assert bool((COND_TABLE[y_wrong, 0] != COND_TABLE[y_eval, 0]).all()), "the wrong text must describe a different drug"
N = len(y_eval)
print("pairs (real validation cells):")
for c in EVAL_CONDS:
    print(f"   {meta['cond_text'][c]}   n={int(counts[c])}")
g = torch.Generator(device=DEV).manual_seed(cfg.seed + 1)
X0 = torch.randn(N, *meta["shape"], device=DEV, generator=g)

def paired_gen():
    '''Fresh generator, same seed: identical per-step noise for every row of a paired comparison.'''
    return torch.Generator(device=DEV).manual_seed(cfg.seed + 2)

base_s = ddim_sample(lambda x, t: base_frozen(x, t, COND(y_eval)), N, meta["shape"],
                     cfg.sample_steps, x_init=X0, generator=paired_gen())
xa_s = ddim_sample(lambda x, t: xa(x, t, TEXT_TOK[y_eval], TEXT_MASK[y_eval], cond_ids=COND(y_eval)),
                   N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen())
xa_wrong = ddim_sample(lambda x, t: xa(x, t, TEXT_TOK[y_wrong], TEXT_MASK[y_wrong], cond_ids=COND(y_eval)),
                       N, meta["shape"], cfg.sample_steps, x_init=X0, generator=paired_gen())
print("sampled:", tuple(base_s.shape), "| NFE per sample:", cfg.sample_steps, "| paired: same X0 for all rows")

SAMPLE_TEXT_DIFF = ((xa_s - xa_wrong).flatten(1).norm(dim=1) / xa_s.flatten(1).norm(dim=1)).mean().item()
print(f"mean relative difference between correct-text and wrong-text samples: {SAMPLE_TEXT_DIFF:.1%}")

# ---- Scale check before any figure or metric: total variance of each row over that of the real
# validation cells. Far above 1 means the row left the data range and nothing computed on it
# means anything.
ROWS = {"base": base_s, "xattn": xa_s, "xattn_wrong_text": xa_wrong}
REAL_VAR = val_loader.dataset.x.var(0).sum().item()
SCALE = {k: v.var(0).sum().item() / REAL_VAR for k, v in ROWS.items()}
SATURATION = {k: saturation(v) for k, v in ROWS.items()}
OUT_OF_RANGE = [k for k, s in SCALE.items() if not math.isfinite(s) or s > 5]
print("variance ratio vs all real validation cells:", {k: round(s, 2) for k, s in SCALE.items()},
      "| on the clamp:", {k: f"{s:.1%}" for k, s in SATURATION.items()})
if OUT_OF_RANGE:
    print(f"\n*** SAMPLES OUT OF RANGE: {OUT_OF_RANGE}. Every number below for these rows is meaningless; "
          "check the base and the sampler before reading on. ***\n")

import matplotlib.pyplot as plt
real_x = val_loader.dataset.x.numpy(); real_c = val_loader.dataset.cond.numpy()
gen_c = y_eval.cpu().numpy()
cmap = plt.get_cmap("tab10")

# Axes fixed by the real cells of the evaluation pairs (1st to 99th percentile, padded) and shared
# by every panel, so an out-of-range row cannot flatten the others; its off-panel share goes in
# the title.
real_sel = np.concatenate([real_x[real_c == c][:PER] for c in EVAL_CONDS])
_lo, _hi = np.percentile(real_sel[:, :2], 1, axis=0), np.percentile(real_sel[:, :2], 99, axis=0)
_pad = 0.5 * (_hi - _lo)
XLIM, YLIM = (_lo[0] - _pad[0], _hi[0] + _pad[0]), (_lo[1] - _pad[1], _hi[1] + _pad[1])

def pair_label(c):
    '''"drug / cell line", not the drug alone: several pairs can share a drug.'''
    t = meta["cond_text"][c]
    return t.split(",")[0] + " / " + t.split(" in ")[-1].replace(" cells", "")

fig, ax = plt.subplots(1, 4, figsize=(19, 4.4), sharex=True, sharey=True)
for a, (name, pts, cs) in zip(ax, [("real (val)", real_x, real_c), ("base, null label", base_s.cpu().numpy(), gen_c),
                                   ("xattn, correct text", xa_s.cpu().numpy(), gen_c),
                                   ("xattn, WRONG text", xa_wrong.cpu().numpy(), gen_c)]):
    n_off = n_pts = 0
    for j, c in enumerate(EVAL_CONDS):
        p = pts[cs == c][:PER]
        a.scatter(p[:, 0], p[:, 1], s=6, alpha=0.5, color=cmap(j), label=pair_label(c) if a is ax[0] else None)
        n_off += int(((p[:, 0] < XLIM[0]) | (p[:, 0] > XLIM[1]) | (p[:, 1] < YLIM[0]) | (p[:, 1] > YLIM[1])).sum())
        n_pts += len(p)
    a.set_title(name + (f"  ({n_off / max(n_pts, 1):.0%} off panel)" if n_off else ""))
    a.set_xlabel("PC1")
ax[0].set_xlim(*XLIM); ax[0].set_ylim(*YLIM)
ax[0].set_ylabel("PC2"); ax[0].legend(fontsize=7, markerscale=2); plt.tight_layout(); plt.show()

## Per-pair metrics

From `src/guidance/eval/cell_metrics.py`, per pair against its real validation cells: MMD, energy
distance and Fréchet distance on the PCs (distance between the two populations), precision and
recall (quality and coverage), the variance ratio (far below 1 is collapse), and drug accuracy
under a k-nearest-neighbour scorer over real training cells that never touched the sampler. The
first block of rows is the real cells themselves; their kNN accuracy is the ceiling that applies
to these unseen pairs, and a generated row that scores above it is over-steered or collapsed, not
better. The wrong-text row is scored against the pair that was requested.

In [ ]:
from guidance.eval.cell_metrics import all_metrics, KNNConditionScorer

DRUG_OF = COND_TABLE[:, 0]                                   # drug id of every condition id
SCORER = KNNConditionScorer(train_loader.dataset.x, DRUG_OF[train_loader.dataset.cond.to(DEV)].cpu(),
                            k=25, seed=cfg.seed, device=DEV)
val_x, val_c = val_loader.dataset.x, val_loader.dataset.cond
SCORER_REAL_ACC = SCORER.accuracy(val_x[:20_000], DRUG_OF[val_c[:20_000].to(DEV)])
print(f"{SCORER.describe()} | drug accuracy on {min(20_000, len(val_x)):,} real validation cells: "
      f"{SCORER_REAL_ACC:.1%} (chance {1 / meta['n_drugs']:.1%})")
gen_c = y_eval.cpu()
METRICS = {"eval_conditions": EVAL_CONDS, "samples_per_condition": PER,
           "scorer": SCORER.describe(), "scorer_real_val_drug_accuracy": SCORER_REAL_ACC}
print(f"\n{'condition':30s} {'arm':18s} {'MMD':>6} {'energy':>7} {'FD_pc':>7} {'prec':>5} {'rec':>5} {'var':>6} {'acc':>6} {'n real':>6}")
# Reference rows: the scorer on each pair's own real validation cells. These pairs are unseen, so
# the reference set holds no cell of them and this is usually far below the all-pairs figure.
# A generated row that scores above it is over-steered or collapsed, not better.
real_rows = []
for c in EVAL_CONDS:
    r = val_x[val_c == c]
    acc = SCORER.accuracy(r[:PER], DRUG_OF[c].repeat(min(PER, len(r))))
    real_rows.append({"cond": int(c), "text": meta["cond_text"][c], "cond_acc": acc, "n_real": int(len(r))})
    print(f"{meta['cond_text'][c][:30]:30s} {'real (val)':18s} {'-':>6} {'-':>7} {'-':>7} {'-':>5} {'-':>5} {'1.00':>6} {acc:6.1%} {len(r):6d}")
METRICS["real_per_condition"] = real_rows
METRICS["real_cond_acc_mean"] = float(np.mean([row["cond_acc"] for row in real_rows]))
for name, s_ in [("base", base_s), ("xattn", xa_s), ("xattn_wrong_text", xa_wrong)]:
    rows = []
    for c in EVAL_CONDS:
        r = val_x[val_c == c]; gsel = s_[gen_c == c]
        m = all_metrics(gsel, r, seed=cfg.seed)
        m["cond_acc"] = SCORER.accuracy(gsel, DRUG_OF[c].repeat(len(gsel)))
        rows.append({"cond": int(c), "text": meta["cond_text"][c], **m})
        print(f"{meta['cond_text'][c][:30]:30s} {name:18s} {m['mmd']:6.3f} {m['energy']:7.3f} {m['frechet_pc']:7.2f} "
              f"{m['precision']:5.2f} {m['recall']:5.2f} {m['var_ratio']:6.2f} {m['cond_acc']:6.1%} {len(r):6d}")
    METRICS[f"{name}_per_condition"] = rows
    for k in ("mmd", "energy", "frechet_pc", "precision", "recall", "var_ratio", "cond_acc"):
        METRICS[f"{name}_{k}_mean"] = float(np.mean([row[k] for row in rows]))
if OUT_OF_RANGE:
    print(f"\n*** rows {OUT_OF_RANGE} left the data range (scale check above): their metrics are not results. ***")

## Manifest

Everything reported above, plus the commit, config, hardware, base source and clamp setting,
written to `manifest_xattn_cells.json`.

In [ ]:
report = {
    "method": "text conditioning by cross-attention (PixArt-alpha block, arXiv:2310.00426)",
    "modality": "cells",
    "code_version": GIT_SHA,
    "config": asdict(cfg),
    "label_path": cfg.label_path,
    "base_source": BASE_SOURCE,
    "x0_clamp": X0_CLAMP_DESC,
    "val_matched_base_same_path": VAL_BASE_SAME,
    "val_matched_base_true_label": VAL_BASE_LABEL,
    "val_matched_xattn": VAL_XA,
    "val_matched_xattn_wrong_text": VAL_XA_WRONG,
    "val_matched_xattn_wrong_text_per_offset": dict(zip(map(str, WRONG_OFFSETS), VAL_XA_WRONG_EACH)),
    "text_effect_rel": TEXT_EFFECT,
    "text_effect_rel_by_noise_bin": TEXT_EFFECT_BY_BIN,
    "sample_text_diff_rel": SAMPLE_TEXT_DIFF,
    "sample_variance_ratio_vs_val": SCALE,
    "samples_out_of_range": OUT_OF_RANGE,
    "on_clamp_fraction": SATURATION,
    "base_best_val_during_training": min(h["val"] for h in base_hist),
    "xattn_best_val_during_training": min(h["val"] for h in xa_hist),
    "trainable_xattn": xa.n_trainable(),
    "frozen_backbone": N_FROZEN,
    "checkpoint_meta": CKPT_META,
    "attach": cfg.attach,
    "nfe": cfg.sample_steps,
    "clip_fallback": IS_FALLBACK,
    "train_seconds": {"base": sum(h.get("seconds", 0) for h in base_hist),
                      "xattn": sum(h.get("seconds", 0) for h in xa_hist)},
    "hardware": torch.cuda.get_device_name(0) if DEV.type == "cuda" else "cpu",
    "device": str(DEV),
}
report.update(globals().get("FID_SCORES", {}))
report.update(globals().get("METRICS", {}))
print(json.dumps(report, indent=2))
with open("manifest_xattn_cells.json", "w") as f:
    json.dump(report, f, indent=2)

## Before reporting any of this

- The scale check flagged nothing (`samples_out_of_range` empty) and `clip_fallback` is false.
- `text_effect_rel` reported with its sign and the per-noise-level breakdown; a small or null
  effect is a result, not a reason to withhold the row.
- MMD and energy distance reported with the variance ratio, never alone; per-pair sample counts
  stated; the validation pairs are unseen (cell line, drug) combinations, say so.
- `condition`, `dose`, `label_path` and `base_source` stated next to every number; the two
  `label_path` settings are different experiments.
- Three seeds (`seed`), mean and standard deviation, with `split_seed` fixed.
- Described as a frozen MLP with a PixArt-α-style residual cross-attention adapter (CLIP not T5,
  one block not every block, one query token, attention-only training, no CFG), not as PixArt-α
  reproduced.